# BERT Fine-Tuning + Sentiment Chatbot

This notebook combines the BERT fine-tuning workflow and the interactive sentiment chatbot.

**What it does**
1. Loads the IMDb movie-review dataset.
2. Tokenizes the text using `bert-base-uncased`.
3. Fine-tunes BERT for positive/negative sentiment classification.
4. Evaluates and saves the fine-tuned model.
5. Loads the saved model and starts an interactive chatbot.

**Note:** This is a sentiment-classification chatbot. It classifies messages as positive or negative; it does not generate open-ended answers like ChatGPT.

Run the cells from top to bottom. Training uses 1,000 training examples and 500 test examples, matching the workflow discussed earlier.

## 1. Install required libraries

In [ ]:
%pip install -q datasets transformers torch accelerate

## 2. Import libraries

In [ ]:
from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline
)
import torch

## 3. Load the IMDb dataset

In [ ]:
dataset = load_dataset("stanfordnlp/imdb")

print(dataset)

## 4. Select training and testing samples

In [ ]:
train_dataset = dataset["train"].select(range(1000))
test_dataset = dataset["test"].select(range(500))

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

## 5. Load the BERT tokenizer

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

## 6. Define the tokenization function

In [ ]:
def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        max_length=256,
        truncation=True
    )

## 7. Define the preprocessing function

In [ ]:
def preprocess(ds):
    ds = ds.map(
        tokenize_fn,
        batched=True,
        remove_columns=["text"]
    )
    ds = ds.rename_column("label", "labels")
    ds.set_format(
        type="torch",
        columns=["input_ids", "attention_mask", "labels"]
    )
    return ds

## 8. Preprocess training and testing data

In [ ]:
train_dataset = preprocess(train_dataset)
test_dataset = preprocess(test_dataset)

print(train_dataset)
print(test_dataset)

## 9. Load the pretrained BERT classification model

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

## 10. Freeze BERT layers and unfreeze the last two layers

In [ ]:
# Freeze all BERT encoder parameters
for param in model.bert.parameters():
    param.requires_grad = False

# Unfreeze the last two encoder layers
for layer in model.bert.encoder.layer[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

# The classification head remains trainable.
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")

## 11. Set training arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./bert-finetuned-imdb",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=10,
    logging_strategy="steps",
    report_to="none"
)

## 12. Create the Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

## 13. Fine-tune BERT

In [ ]:
train_result = trainer.train()
print(train_result)

## 14. Evaluate the fine-tuned model

In [ ]:
metrics = trainer.evaluate()
print(metrics)

## 15. Save the fine-tuned model and tokenizer

In [ ]:
MODEL_PATH = "./bert-finetuned-imdb"

trainer.save_model(MODEL_PATH)
tokenizer.save_pretrained(MODEL_PATH)

print("Model and tokenizer saved to:", MODEL_PATH)

## 16. Reload the saved fine-tuned model

In [ ]:
tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)
model = BertForSequenceClassification.from_pretrained(MODEL_PATH)

device = 0 if torch.cuda.is_available() else -1

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=device
)

print("Fine-tuned model loaded successfully!")

## 17. Test the model with one review

In [ ]:
sample_text = "This movie was amazing and I loved the acting!"

result = classifier(sample_text, truncation=True, max_length=256)
print(result)

## 18. Define the sentiment chatbot

IMDb labels are `0 = negative` and `1 = positive`. The function below handles the standard Hugging Face labels `LABEL_0` and `LABEL_1`.

In [ ]:
def get_sentiment_label(prediction):
    label = str(prediction["label"]).lower()

    if "positive" in label:
        return "positive"
    if "negative" in label:
        return "negative"
    if label == "label_1":
        return "positive"
    if label == "label_0":
        return "negative"

    raise ValueError(f"Unrecognized model label: {prediction['label']}")


def chatbot():
    print("\\n===================================")
    print("       BERT SENTIMENT CHATBOT")
    print("===================================")
    print("Enter a message for sentiment analysis.")
    print("Type 'exit' to stop.\\n")

    while True:
        user_input = input("You: ").strip()

        if user_input.lower() in {"exit", "quit", "bye"}:
            print("Bot: Goodbye! Have a nice day.")
            break

        if not user_input:
            print("Bot: Please enter a message.\\n")
            continue

        prediction = classifier(
            user_input,
            truncation=True,
            max_length=256
        )[0]

        sentiment = get_sentiment_label(prediction)
        confidence = prediction["score"] * 100

        if sentiment == "positive":
            print("Bot: Sentiment: Positive 😊")
            print(f"Bot: Confidence: {confidence:.2f}%")
            print("Bot: That's great! Your message sounds positive.\\n")
        else:
            print("Bot: Sentiment: Negative 😔")
            print(f"Bot: Confidence: {confidence:.2f}%")
            print("Bot: I'm sorry to hear that.\\n")

## 19. Start chatting

In [ ]:
chatbot()

## 20. Optional: Test several messages

In [ ]:
sample_messages = [
    "I really enjoyed this movie. It was amazing!",
    "This movie was very boring and disappointing.",
    "The acting was excellent and the story was wonderful."
]

for message in sample_messages:
    prediction = classifier(
        message,
        truncation=True,
        max_length=256
    )[0]

    sentiment = get_sentiment_label(prediction)
    print("Text:", message)
    print("Sentiment:", sentiment)
    print(f"Confidence: {prediction['score'] * 100:.2f}%")
    print("-" * 60)

## Notes

- The first training run downloads the IMDb dataset and pretrained BERT weights, so an internet connection is required.
- Training is performed on 1,000 examples for one epoch to keep this teaching demonstration manageable. Increase the sample size and epochs for a more thorough experiment.
- The `bert-finetuned-imdb` folder is created when the model is saved. Keep it if you want to use the chatbot later without retraining.
- If you restart the notebook kernel, run the installation/import cells and then load the saved model from `./bert-finetuned-imdb` before starting the chatbot.